### SETUP

In [ ]:
cd ..

In [ ]:
pwd

In [3]:
import torch, os, re
import numpy as np
import matplotlib.pyplot as plt
import visualizations.plot_settings as plot_settings

In [4]:
plot_settings.set_latex_settings()

### SETTINGS

In [5]:
experiment_name = "faq"
bo_filtering_dir_path = f"./experiments/baselines/results/{experiment_name}"
tosfit_dir_path = f"./experiments/tosfit/results/{experiment_name}"

In [6]:
metric = 'simple_reward'
metric_label = "Simple Reward" #"Simple Reward"
sign = 1 # negative sign converts rewards to -rewards, which can be understood as a notion of regret

In [7]:
discriminative_config_entry = "learning_rate"
batch_efficiency = False
num_runs_to_include = 1000 # ensures all runs get equally many seeds counted
equalize_samples_from_dataset = False # ensures each entry of the dataset has equally many seeds, effectively disables sampling from dataset and rather reports on the mean score across the dataset

#### Relabeling and filtering settings

In [ ]:
settings_labels = {
    "qwen3_embedding_0.6B-normalize-bias-IT-4.0": 'Unguided Generation',
    "mte-normalize-bias-IT-4.0": 'Unguided Generation',
    "pauli_observables-bias-IT-16.0": 'Unguided Generation',
    "qwen3_embedding_0.6B-normalize-bias-TS-4.0": 'Post-Generation TS',
    "mte-normalize-bias-TS-4.0": 'Post-Generation TS',
    "pauli_observables-bias-TS-4.0": 'Post-Generation TS',
    "tosfit-1e-06": "ToSFiT 1E-6",
    "tosfit-1e-07": "ToSFiT 1E-7",
    "tosfit-1e-08": "ToSFiT 1E-8",
    "tosfit-1": "ToSFiT 1",
    "tosfit-4": "ToSFiT 4",
    "tosfit-16": "ToSFiT 16",
}

# ['Generate \& Evaluate', 'Generate All \& TS', 'ToSFiT', 
accepted_settings = ['Unguided Generation', 'ToSFiT 1E-6', 'ToSFiT 1E-7', 'ToSFiT 1E-8']
#...

filters = {
    #'learning_rate': 2e-05,
}

#### Plot setup

In [9]:
figwidth, figheight = plot_settings.column_width, 2.0

In [10]:
y_limits = (0.74, 0.85) #500.0)
x_max = 1000
x_scale = "linear"
y_scale = "linear"

In [11]:
x_label = "\# Evaluation Batches" if batch_efficiency else "\# Evaluations"

In [12]:
x_subsampling_number = 1 # must be positive integer (helps with memory issues due to too many datapoints)

In [13]:
plot_name = "faq_simple_reward_full"

In [ ]:
legend_order = [
    'Unguided Generation',
    'Post-Generation TS',
    'ToSFiT',
    'ToSFiT 1E-6',
    'ToSFiT 1E-7',
    'ToSFiT 1E-8',
    'ToSFiT 1',
    'ToSFiT 4',
    'ToSFiT 16',
]
legend_location = 'lower right'
legend = True

### Load Data

In [15]:
metrics = {}

#### Baselines 

In [16]:
for root, _, files in os.walk(bo_filtering_dir_path):  # Recursively traverse directories
    for filename in files:
        if filename.endswith("-metrics.pt"):
            filepath = os.path.join(root, filename)
            data = torch.load(filepath, weights_only=False, map_location='cpu')
            setting_name = re.match(r'\d+-[^\d]+-\d+-(.*)-metrics\.pt', filename).group(1)
            if setting_name not in metrics:
                metrics[setting_name] = [] if not equalize_samples_from_dataset else {}
            if equalize_samples_from_dataset:
                prompt_sample = data['config']['prompt_sample']
                if prompt_sample not in metrics[setting_name]:
                    metrics[setting_name][prompt_sample] = []
                metrics[setting_name][prompt_sample].append(data[metric])
            else:
                metrics[setting_name].append(data[metric])

#### tosfit

In [17]:
for root, _, files in os.walk(tosfit_dir_path):  # Recursively traverse directories
    for filename in files:
        if filename.endswith("-metrics.pt"):
            filepath = os.path.join(root, filename)
            data = torch.load(filepath, weights_only=False, map_location='cpu')
            for key, value in filters.items():
                if data['config'][key] != value:
                    continue
            setting_name = "tosfit-" + str(data['config'][discriminative_config_entry])# os.path.basename(root)
            metric_to_add = data[metric]
            if not batch_efficiency:
                metric_to_add = metric_to_add.repeat_interleave(data['config']['bo_batch_size'])
            if setting_name not in metrics:
                metrics[setting_name] = [] if not equalize_samples_from_dataset else {}
            if equalize_samples_from_dataset:
                prompt_sample = data['config']['prompt_sample']
                if prompt_sample not in metrics[setting_name]:
                    metrics[setting_name][prompt_sample] = []
                metrics[setting_name][prompt_sample].append(metric_to_add)
            else:
                metrics[setting_name].append(metric_to_add)
            

In [18]:
"""
for single_setting_dict in metrics.values():
    for single_prompt_list in single_setting_dict.values():
        print(len(single_prompt_list))
"""

'\nfor single_setting_dict in metrics.values():\n    for single_prompt_list in single_setting_dict.values():\n        print(len(single_prompt_list))\n'

In [19]:
if equalize_samples_from_dataset:
    least_seeds = min(len(single_prompt_list) for single_setting_dict in metrics.values() for single_prompt_list in single_setting_dict.values())
    
    print(f"The total number of seeds when averaging across the dataset is {least_seeds}.")
    metrics = {setting_name: {prompt: prompt_list[:least_seeds] for prompt, prompt_list in single_setting_dict.items()} for setting_name, single_setting_dict in metrics.items()}
    metrics = {setting_name: [sum(single_seed_metrics)/len(single_seed_metrics) for single_seed_metrics in zip(*list(single_setting_dict.values()))] for setting_name, single_setting_dict in metrics.items()}

##### Relabel settings

In [20]:
metrics = {settings_labels.get(key, key): value for key, value in metrics.items()}

##### Filter settings

In [21]:
metrics = {key: value for key, value in metrics.items() if accepted_settings is ... or key in accepted_settings}

### Plot Statistics

In [22]:
figure1, ax1 = plt.subplots(1, 1, figsize=(figwidth, figheight))

ax1.set_xlabel(x_label)
ax1.set_ylabel(metric_label)
ax1.set_ylim(y_limits)
ax1.set_xscale(x_scale)
ax1.set_yscale(y_scale)

In [23]:
for setting_name in metrics.keys():
    all_metrics = sign * np.stack([best_rewards.numpy(force=True) for best_rewards in metrics[setting_name]])[:num_runs_to_include, :]
    n_samples = all_metrics.shape[0]
    print(f"{n_samples} samples for {setting_name}")

    mean_metrics = np.mean(all_metrics, axis=0)[:x_max]
    std_metrics = np.std(all_metrics, axis=0, ddof=1)[:x_max]
    stderr_metrics = (std_metrics / n_samples**.5)[:x_max]
    
    x_values = np.arange(1, mean_metrics.size+1)[:x_max]

    ax1.plot(x_values, mean_metrics, label=setting_name, **plot_settings.format(setting_name))
    ax1.fill_between(x_values, mean_metrics + stderr_metrics, mean_metrics - stderr_metrics, alpha=0.2, linewidth=0.000001, **plot_settings.format(setting_name))

if legend:
    if legend_order is not None:
        handles, labels = ax1.get_legend_handles_labels() 
        order1 = [labels.index(x) for x in legend_order if x in metrics.keys()]
        ax1.legend([handles[i] for i in order1], [labels[i] for i in order1], loc=legend_location)
    else:
        ax1.legend(loc=legend_location)

25 samples for Unguided Generation
25 samples for GP-tosfit 1E-7
25 samples for GP-tosfit 1E-6
25 samples for GP-tosfit 1E-8


In [ ]:
figure1.savefig(f"visualizations/results/{plot_name}.pgf", bbox_inches="tight")

In [ ]:
cd visualizations

In [ ]:
! bash pgf_compiler.sh {plot_name}


Compiling PGF plots to PDF...
[1/1] Compiled faq_simple_reward_full.pgf to PDF. Progress: 100% 
All PGF plots have been compiled to PDF.
